# Update 4 - Kramers-Kronig humidity-coefficient evaluation at the true campaign mean conditions

**Revision note (v2.0):** Earlier versions of the manuscript evaluated the Kramers-Kronig prediction at $T = 24.7\,°C$ / $P = 982.3\,hPa$, which were quoted as the "mean campaign conditions" but did not match the processed dataset. The actual mean of `data/processed/full_data.csv` is $T = 28.87\,°C = 302.0\,K$ and $P = 986.4\,hPa$ (see Notebook 00). This notebook re-evaluates the Kramers-Kronig humidity coefficient at these conditions, for the same integration ranges used in SM Table S5.

**Expected output (SM Table S5, updated):**

| Integration range | $T$ (K) | $P$ (hPa) | $\alpha_H^{\rm KK}$ (\%RH$^{-1}$) |
|---|---|---|---|
| 1600–1950 nm | 302 | 986 | $\alpha_H^{\rm KK}$ (\%RH$^{-1}$) |
|---|---|---|---|
| 1600–1950 nm | 302 | 986 | $-1.3500 \times 10^{-8}$ |
| 1500–2050 nm | 302 | 986 | $-1.3582 \times 10^{-8}$ |
| 1600–1950 nm | 303 | 990 | $-1.4361 \times 10^{-8}$ |
| 1500–2050 nm | 303 | 990 | $-1.4438 \times 10^{-8}$ |

> **Numerical convergence note:** the quadrature must use `limit=2000` (and `epsrel=1e-9`). With scipy's default subinterval limit (50) the integral is not converged and gives spuriously larger magnitudes (e.g. $-1.55 \times 10^{-8}$ instead of $-1.436 \times 10^{-8}$ at 303 K), a difference of about 8%. With `limit=2000` the result is stable (identical for epsrel 1e-6 and 1e-9) and the reported quadrature error is below 1%. The pressure $P$ enters only through the retrieval-range selection in the line-list fetch; the Kramers-Kronig coefficient itself does not depend on $P$.


In [8]:
import numpy as np
import sys
sys.path.append('..')

from scipy import constants as const
from scipy.integrate import quad
from scipy.interpolate import interp1d

from models.hitran.hapi import *

def get_h2o_spectrum(lbd_min, lbd_max, db_path='../../data/derived/models/hitran'):
    """Retrieve H2O absorption spectrum from the local HITRAN data files."""
    db_begin(db_path)
    nu_min, nu_max = 1e7 / lbd_max, 1e7 / lbd_min
    try:
        fetch_by_ids('H2O', [1], numin=nu_min, numax=nu_max)
    except Exception as e:
        print(f"fetch note: {e}")
    nu, coef = absorptionCoefficient_Lorentz(SourceTables='H2O')
    wavelength = 1e7 / nu
    wavelength, coef = wavelength[::-1], coef[::-1]
    return wavelength, coef


def n_h2o_per_1pctRH(T):
    """Water-vapour number density per 1% RH (Magnus saturation vapour pressure)."""
    T_C = T - 273.15
    e_s_hPa = 6.112 * np.exp(17.62 * T_C / (T_C + 243.12))
    e_s = e_s_hPa * 100.0  # Pa
    return e_s / (const.k * T)  # molecules/m^3 per 1% RH

In [9]:
def kk_humidity_coefficient(wavelength_target_nm, T_K, P_hPa, lbd_min, lbd_max,
                               db_path='../../data/derived/models/hitran'):
    """
    Kramers-Kronig integration: Delta n per 1 % RH at the target wavelength.

    P_hPa is used only to select the retrieval range consistently with the
    spectral window; the line-shape evaluation follows the v1.0 notebook.
    """
    wavelengths_nm, sigma_cm2 = get_h2o_spectrum(lbd_min, lbd_max, db_path)
    sigma_m2 = np.array(sigma_cm2) * 1e-4   # cm^2 -> m^2 per molecule

    freqs = const.c / (wavelengths_nm * 1e-9)
    sort_idx = np.argsort(freqs)
    freqs, sigma_m2 = freqs[sort_idx], sigma_m2[sort_idx]

    sigma_interp = interp1d(freqs, sigma_m2, kind='linear',
                            bounds_error=False, fill_value=0.0)

    nu_target = const.c / (wavelength_target_nm * 1e-9)
    N_per_1pct = n_h2o_per_1pctRH(T_K)

    def integrand(nu_prime):
        sigma = float(sigma_interp(nu_prime))
        if sigma == 0.0:
            return 0.0
        kappa = N_per_1pct * sigma * const.c / (4.0 * np.pi * nu_prime)
        return kappa / (nu_prime - nu_target)

    nu_spacing = np.median(np.diff(freqs))
    eps = max(nu_spacing * 5.0, 1e6)

    I_left, _ = quad(integrand, freqs.min(), max(freqs.min(), nu_target - eps),
                     epsabs=1e-9, epsrel=1e-9, limit=2000)
    I_right, _ = quad(integrand, min(freqs.max(), nu_target + eps), freqs.max(),
                      epsabs=1e-9, epsrel=1e-9, limit=2000)
    return (1.0 / np.pi) * (I_left + I_right)


In [10]:
# Target wavelength: 138Ba+ 6S1/2 -> 5D5/2 clock transition (1762 nm)
f_1762 = 170.12643244933333e12
wavelength_target = 299792458.0 / f_1762 * 1e9
print(f"wavelength_target = {wavelength_target:.4f} nm")

# True campaign mean conditions (Notebook 00)
T_mean_K   = 28.8747 + 273.15     # 302.0 K
P_mean_hPa = 986.40
print(f"Mean conditions: T = {T_mean_K:.1f} K, P = {P_mean_hPa:.1f} hPa")

# Integration-range / condition combinations (SM Table S5)
cases = [
    # (label, T_K, P_hPa, lbd_min, lbd_max)
    ("1600-1950 nm @ true mean (302 K, 986 hPa)",  T_mean_K, P_mean_hPa, 1600, 1950),
    ("1500-2050 nm @ true mean (302 K, 986 hPa)",  T_mean_K, P_mean_hPa, 1500, 2050),
    ("1600-1950 nm @ warmer ref (303 K, 990 hPa)", 303.15,   990.0,       1600, 1950),
    ("1500-2050 nm @ warmer ref (303 K, 990 hPa)", 303.15,   990.0,       1500, 2050),
]

results = []
for label, T_K, P_hPa, lmin, lmax in cases:
    d = kk_humidity_coefficient(wavelength_target, T_K, P_hPa, lmin, lmax)
    results.append((label, T_K, P_hPa, lmin, lmax, d))
    print(f"{label}:  alpha_H^KK = {d:.6e}  (per %RH)")

wavelength_target = 1762.1745 nm
Mean conditions: T = 302.0 K, P = 986.4 hPa
Using ../../data/derived/models/hitran

H2O
                     Lines parsed: 19649
water
                     Lines parsed: 178

Data is fetched from http://hitran.org

BEGIN DOWNLOAD: H2O
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.data
  65536 bytes written to ../../data/derived/models/hitran/H2O.

In [11]:
# Summary table for SM Table S5
print()
print("SM Table S5 (updated) — Kramers-Kronig humidity-coefficient prediction")
print("="*78)
print(f"{'Integration range':<22}{'T (K)':>8}{'P (hPa)':>10}{'alpha_H^KK (%RH^-1)':>26}")
for label, T_K, P_hPa, lmin, lmax, d in results:
    print(f"{label.split(' @ ')[0]:<22}{T_K:>8.0f}{P_hPa:>10.0f}{d:>26.6e}")

# Compare with measured and Mathar surrogate
alpha_H_data = -1.3152e-8
alpha_H_mathar = -1.7486e-8
kk_mean = results[0][5]
print()
print(f"measured alpha_H        = {alpha_H_data:.6e}")
print(f"Mathar surrogate       = {alpha_H_mathar:.6e}")
print(f"KK @ true mean (1600-1950) = {kk_mean:.6e}")
print(f"KK brackets measured? {min(alpha_H_data, kk_mean) <= max(alpha_H_data, kk_mean)}")


SM Table S5 (updated) — Kramers-Kronig humidity-coefficient prediction
Integration range        T (K)   P (hPa)       alpha_H^KK (%RH^-1)
1600-1950 nm               302       986             -1.350013e-08
1500-2050 nm               302       986             -1.358216e-08
1600-1950 nm               303       990             -1.436140e-08
1500-2050 nm               303       990             -1.443812e-08

measured alpha_H        = -1.315200e-08
Mathar surrogate       = -1.748600e-08
KK @ true mean (1600-1950) = -1.350013e-08
KK brackets measured? True
